# 03 String Formatting

Review and standardize inconsistent data formats across 95,000 employee records.

The goal is to ensure consistent category labels, clean text fields, standardize date formats, and convert salary values into a consistent numerical format without changing their meaning.m

## 1. Preparation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/02_missing_values_resolved.csv")
df_clean = df.copy()

text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Text columns: {len(text_columns)}")

Rows: 95,000
Columns: 32
Text columns: 15


## 2. Detection

In [2]:
for col in text_columns:
    print(f"\n{'=' * 60}")
    print(f"{col} | Unique values: {df[col].nunique():,}")
    print(f"{'=' * 60}")

    display(
        df[col]
        .value_counts(dropna=False)
        .head(50)
        .to_frame("count")
    )


employee_id | Unique values: 95,000


,count
employee_id,
E145770,1
E162489,1
E143182,1
E136944,1
E188443,1
E175804,1
LEG-65921,1
E108429,1
E174881,1



full_name | Unique values: 68,153


,count
full_name,
Michael Smith,51
John Smith,32
David Williams,30
Michael Brown,29
James Smith,29
James Johnson,27
Michael Jones,27
Michael Johnson,27
Jennifer Smith,27



source_dataset | Unique values: 3


...

,count
currency,
USD,16708
EUR,16645
GBP,11215
KES,5730
ZAR,5662
PHP,5644
AUD,5629
INR,5614
CAD,5559



salary_raw | Unique values: 93,351


,count
salary_raw,
31946,4
60819,3
"$30,638",3
"$35,945",3
"€44,126",3
41028,3
27310,3
36446,3
38987,3



pay_band | Unique values: 5


,count
pay_band,
Band 2,37703
Band 1,36855
Band 3,18720
Band 4,1707
Band 5,15


## 3. Text and Date Standardization

Standardize gender, department, and country labels, remove unnecessary whitespace from job titles, and convert hire dates to a consistent format (YYYY-MM-DD).

In [3]:
gender_mapping = {
    "male": "Male",
    "man": "Male",
    "m": "Male",
    "female": "Female",
    "woman": "Female",
    "f": "Female",
    "non-binary": "Non-binary",
    "nb": "Non-binary",
    "other": "Other",
    "prefer not to say": "Prefer not to say"
}

department_labels = [
    "HR", "IT", "Operations", "Design",
    "Marketing", "Engineering", "Finance",
    "Customer Support", "Sales", "Legal",
    "Data Science", "Product"
]

department_mapping = {
    value.casefold(): value
    for value in department_labels
}

country_mapping = {
    "US": "United States",
    "USA": "United States",
    "UK": "United Kingdom"
}

gender_normalized = df_clean["gender"].astype("string").str.strip().str.casefold()
department_normalized = df_clean["department"].astype("string").str.strip().str.casefold()

assert gender_normalized.dropna().isin(gender_mapping).all()
assert department_normalized.dropna().isin(department_mapping).all()

df_clean["gender"] = gender_normalized.map(gender_mapping)
df_clean["department"] = department_normalized.map(department_mapping)
df_clean["country"] = df_clean["country"].replace(country_mapping)

df_clean["job_title"] = (
    df_clean["job_title"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

hire_dates = df_clean["hire_date"].astype("string").str.strip()

date_formats = {
    r"\d{2}/\d{2}/\d{4}": "%d/%m/%Y",
    r"\d{2}-\d{2}-\d{4}": "%m-%d-%Y",
    r"\d{4}-\d{2}-\d{2}": "%Y-%m-%d",
    r"\d{4}/\d{2}/\d{2}": "%Y/%m/%d",
    r"\d{1,2} [A-Za-z]{3} \d{4}": "%d %b %Y"
}

parsed_dates = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns]")

for pattern, date_format in date_formats.items():
    mask = hire_dates.str.fullmatch(pattern).fillna(False)

    parsed_dates.loc[mask] = pd.to_datetime(
        hire_dates.loc[mask],
        format=date_format,
        errors="coerce"
    )

assert parsed_dates.notna().all(), "Unrecognized or invalid hire dates found."

df_clean["hire_date"] = parsed_dates.dt.strftime("%Y-%m-%d")

for column in ["gender", "department", "country", "job_title", "hire_date"]:
    changed_mask = (
        df[column].astype("string").fillna("<MISSING>")
        != df_clean[column].astype("string").fillna("<MISSING>")
    )

    print(f"\n{column}: {changed_mask.sum():,} values standardized")

    if column in ["gender", "department", "country"]:
        display(
            df_clean[column]
            .value_counts(dropna=False)
            .to_frame("count")
        )

    elif column == "job_title":
        display(
            pd.DataFrame({
                "original": df.loc[changed_mask, column].head(10).map(repr),
                "cleaned": df_clean.loc[changed_mask, column].head(10).map(repr)
            })
        )

    else:
        display(
            pd.DataFrame({
                "original": df.loc[changed_mask, column].head(10),
                "cleaned": df_clean.loc[changed_mask, column].head(10)
            })
        )


gender: 72,266 values standardized


,count
gender,
Male,47859
Female,40548
NaN,2865
Non-binary,2204
Other,763
Prefer not to say,761



department: 943 values standardized


,count
department,
HR,8061
IT,8020
Operations,7994
Marketing,7932
Design,7932
Engineering,7923
Finance,7913
Customer Support,7910
Legal,7866



country: 16,819 values standardized


,count
country,
United States,16708
United Kingdom,11215
Kenya,5730
Germany,5708
South Africa,5662
Philippines,5644
Australia,5629
India,5614
Canada,5559



job_title: 2,856 values standardized


,original,cleaned
25,' SENIOR DATA SCIENTIST ','SENIOR DATA SCIENTIST'
74,' SALES REP ','SALES REP'
95,' CONTENT STRATEGIST ','CONTENT STRATEGIST'
98,' OPERATIONS MANAGER ','OPERATIONS MANAGER'
102,' ML ENGINEER ','ML ENGINEER'
158,' ENGINEERING MANAGER ','ENGINEERING MANAGER'
173,' CFO ','CFO'
181,' SUPPORT TEAM LEAD ','SUPPORT TEAM LEAD'
208,' CMO ','CMO'
241,' CUSTOMER SUCCESS MANAGER ','CUSTOMER SUCCESS MANAGER'



hire_date: 76,174 values standardized


,original,cleaned
0,2026/05/12,2026-05-12
1,2024/12/05,2024-12-05
4,30 Apr 2019,2019-04-30
5,24/02/2026,2026-02-24
6,04-15-2019,2019-04-15
8,2026/06/17,2026-06-17
11,09 Jun 2023,2023-06-09
12,31/03/2024,2024-03-31
14,06-16-2024,2024-06-16
17,24 Apr 2025,2025-04-24


## 4. Salary Formatting

Identify salary values containing currency symbols, currency codes, or other non-numeric characters.

Validate currency consistency and convert salary amounts into numerical values without changing their original currency.

In [4]:
mask = df_clean["salary_raw"].astype("string").str.contains(
    r"[^0-9.,]", regex=True, na=False
)

print(f"Records with additional characters: {mask.sum():,}")

sample = df_clean.loc[
    mask, ["salary_raw", "currency"]
].sample(n=min(100, mask.sum()), random_state=42)

with pd.option_context("display.max_rows", 100):
    display(sample)

Records with additional characters: 32,950


,salary_raw,currency
59537,"€20,422",EUR
12608,"1,927,853 PHP",PHP
60520,"2,252,558 INR",INR
13490,"$72,248",USD
13481,"2,841,438 PHP",PHP
30081,"83,556,338 NGN",NGN
39279,"64,394,422 NGN",NGN
32487,"86,194 SGD",SGD
51216,"68,449 SGD",SGD
15615,"52,979 CAD",CAD


Standardize salary values by removing currency symbols, currency codes, and thousands separators.

In [5]:
salary = df_clean["salary_raw"].astype("string").str.strip()

parts = salary.str.extract(
    r"^(?P<symbol>[$€£])?\s*(?P<amount>(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)(?:\s+(?P<code>[A-Z]{3}))?$"
)

symbol_mapping = {
    "$": "USD",
    "€": "EUR",
    "£": "GBP"
}

symbol_currency = parts["symbol"].map(symbol_mapping)

currency_mismatch = (
    (parts["symbol"].notna() & symbol_currency.ne(df_clean["currency"]))
    | (parts["code"].notna() & parts["code"].ne(df_clean["currency"]))
)

ambiguous_dots = parts["amount"].str.fullmatch(
    r"\d{1,3}(?:\.\d{3})+"
).fillna(False)

print(f"Unrecognized formats: {parts['amount'].isna().sum():,}")
print(f"Currency mismatches: {currency_mismatch.sum():,}")
print(f"Ambiguous decimal formats: {ambiguous_dots.sum():,}")

assert parts["amount"].notna().all()
assert not currency_mismatch.any()
assert not ambiguous_dots.any()

df_clean["salary_raw"] = pd.to_numeric(
    parts["amount"].str.replace(",", "", regex=False)
)

print(f"\nStandardized salaries: {len(df_clean):,}")

sample = df.sample(10, random_state=42)

display(
    pd.DataFrame({
        "original": sample["salary_raw"],
        "cleaned": df_clean.loc[sample.index, "salary_raw"],
        "currency": sample["currency"]
    })
)

Unrecognized formats: 0
Currency mismatches: 0
Ambiguous decimal formats: 0

Standardized salaries: 95,000


,original,cleaned,currency
43450,38896,38896.0,CAD
14791,"33,708.93",33708.93,EUR
28617,"82,531.84",82531.84,AUD
60875,"26,479.62",26479.62,GBP
50419,70370,70370.0,USD
79901,"55,951.30",55951.3,AUD
91111,1228340,1228340.0,ZAR
34188,"36,224.57",36224.57,GBP
18106,440754,440754.0,ZAR
56770,"$26,375",26375.0,USD


## 5. Export

In [6]:
from pathlib import Path

assert len(df_clean) == len(df)
assert df_clean["salary_raw"].notna().all()
assert df_clean["currency"].equals(df["currency"])

output_path = Path("data/processed/03_string_formatting.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")

Saved: data/processed/03_string_formatting.csv
Rows: 95,000
Columns: 32


## 6. Summary

- Standardized gender labels from 15 to 5 categories.
- Standardized department labels from 24 to 12 categories.
- Standardized country names and abbreviations.
- Removed unnecessary whitespace from 2,856 job titles.
- Standardized 76,174 hire dates to YYYY-MM-DD format.
- Converted salary values into numerical format.
- Verified salary currency consistency.

### Final Dataset

- **95,000** employee records.
- **32** columns.
- All validation checks passed.
- Ready for the next cleaning stage.